# CN code extraction from Council Regulation (EU) No 833/2014
Downloads every consolidated version of the regulation through the EUR-Lex API, parses the annex tables and writes one CSV of CN codes per version, with a quality report per version and cross-version checks against the HS nomenclature. The later cells are diagnostics for "ex" codes, graph-ready summaries, and a diff harness used to compare two runs of the extractor. Written for Google Colab.

Required packages

In [ ]:
# Cell 1 — install (run once)
!pip install eurlex beautifulsoup4 pandas openpyxl

print()
print("All done!")

Script for:

1. getting the CELEX ID of each version of the sanctions law

2. getting a .csv per version of the law collecting the sanctioned codes with Annex placement and description

3. performing per-version quality checks

4. producing two cross-version artefacts:
   - `RU-NOT_IN_REFERENCE.csv` — codes the validator could not place in the right HS vintage, with an `is_catch_all` flag distinguishing EU CN catch-all forms (like `ex2106 00 00`) from genuine missing codes (like deprecated `441294` / `441860`), and an `n_versions_not_in_ref` column for persistence.
   - `RU-EX_EVOLUTION.csv` — per-version counts of each HS digit level (HS2..HS10 + ex variants). HS2/exHS2 columns are present and fill with zeros for versions that don't use chapter-level codes (the 24 April 2026 amendment introduced "88" Aircraft in Annex XI).

   *(The previous annex-stability / annex-changes / ex-transitions / ex-redundancy aggregates were removed on 2026-05-20. They collapsed each CN code into a single per-version state, which is incompatible with the direction-aware analysis built downstream from the Python article-direction parser. When the future "treatment over time, split by import vs export" analysis is needed, it will be written fresh against the panel × parser join.)*

To work, it requires 2 input files in `/content/`:

- `RU-CelexIDs_da_ripulire_34.txt`
- `GENERAL-6-digits_codes_2012_2017_2022.csv`


In [4]:
# ============================================================
# CN CODE EXTRACTOR — EU Regulation 833/2014 (Russia)  [v7: +B.13 chapters]
# ============================================================
# Inputs (Colab):
#   - RU-CelexIDs_da_ripulire_34.txt
#   - GENERAL-6-digits_codes_2012_2017_2022.csv
#
# Output folder (Colab): RU-CN_results_<timestamp>/
#   contains, for each of the 34 versions:
#     RU-CN_extraction_YYYY-MM-DD.csv  the extracted CN codes
#     RU-CN_quality_YYYY-MM-DD.txt     the quality check report
#   plus three cross-version artefacts:
#     RU-NOT_IN_REFERENCE.csv          codes the validator could not
#                                      locate in the right HS vintage,
#                                      with is_catch_all flag and
#                                      n_versions_not_in_ref column
#     RU-EX_EVOLUTION.csv              per-version count of each HS
#                                      digit level (HS2..HS10) and
#                                      its ex variant
#     RU-SUMMARY_run_<timestamp>.csv   one row per version
#
# REMOVED (2026-05-20): the cross-version annex-stability /
# annex-changes / ex-transitions / ex-redundancy analysis (former
# Section 5 + part of Section 7). That analysis collapsed each CN
# code into a single per-version state, which is incompatible with
# the direction-aware view we now build downstream (annex carries
# import-vs-export semantics; a "moved" code may represent two
# different legal regimes, not one moving regime). When the future
# import-vs-export-over-time analysis becomes concrete, write it
# fresh against the R Step 3 panel × Python parser output, where
# direction is already a column.
# ============================================================

from bs4 import BeautifulSoup
import pandas as pd
import re, os, time, shutil
from datetime import datetime
from eurlex import get_html_by_celex_id
from google.colab import files


# ============================================================
# SECTION 1 — SETUP
# ============================================================

run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_folder = f"/content/RU-CN_extraction_results/RU-CN_results_{run_timestamp}"
os.makedirs(output_folder, exist_ok=True)

print("=" * 60)
print("CN Code Extractor — EU Regulation 833/2014 — All Versions")
print("=" * 60)
print(f"\nOutput folder: {output_folder}\n")

# ── Load CN reference list (3 vintages) ──────────────────────
cn_ref = pd.read_csv("/content/GENERAL-6-digits_codes_2012_2017_2022.csv", dtype=str)

cn_ref_sets = {
    v: set(
        cn_ref[f"Y{v}"].dropna()
        .str.strip().str.replace(r'["\s]', "", regex=True).str.zfill(6)
    )
    for v in ("2012", "2017", "2022")
}

print("CN reference codes loaded:")
for v, s in cn_ref_sets.items():
    print(f"  Y{v}: {len(s)} codes")


def pick_vintage(date_yyyymmdd):
    """Return HS vintage in force on a YYYY-MM-DD date."""
    if date_yyyymmdd < "2017-01-01": return "2012"
    if date_yyyymmdd < "2022-01-01": return "2017"
    return "2022"


# ── Extract CELEX IDs (oldest → newest) ──────────────────────
with open("/content/RU-CelexIDs_da_ripulire_34.txt", encoding="utf-8") as f:
    raw_html = f.read()

celex_ids = re.findall(r'data-celex="([\w-]+)"', raw_html)
celex_ids = [c for c in celex_ids if c.startswith("02014R0833")]
celex_ids.append("32014R0833")          # original act, manual
celex_ids = celex_ids[::-1]             # reverse → oldest first

print(f"Versions to process      : {len(celex_ids)}\n")


# ============================================================
# SECTION 2 — HELPERS
# ============================================================

# WIN #1: a single parser used everywhere we previously stripped "ex" /
# extracted digits. Returns (has_ex, digits_no_spaces, n_digits).
EX_RX = re.compile(r"^ex\s*", re.IGNORECASE)

def parse_code(code):
    """Return (has_ex: bool, digits: str without spaces, n: int)."""
    code   = code.strip()
    has_ex = bool(EX_RX.match(code))
    digits = re.sub(r"\s+", "", EX_RX.sub("", code))
    return has_ex, digits, len(digits)


def normalize_cn(raw_code):
    """Normalize to 'XXXX XX XX' form, no space after ex (e.g. ex7304 11 00)."""
    has_ex, d, n = parse_code(raw_code)
    if   n <= 4: out = d
    elif n <= 6: out = f"{d[:4]} {d[4:]}"
    elif n <= 8: out = f"{d[:4]} {d[4:6]} {d[6:]}"
    else:        out = f"{d[:4]} {d[4:6]} {d[6:8]} {d[8:]}"
    return f"ex{out}" if has_ex else out


def classify_cn(code):
    """HS4 / HS6 / HS8 / HS10 / exHS* / unknown."""
    has_ex = bool(re.match(r"^ex", code, re.IGNORECASE))
    n      = len(re.sub(r"\D", "", code))
    label  = {2: "HS2", 4: "HS4", 6: "HS6", 8: "HS8", 10: "HS10"}.get(n, "unknown")  # B.13: HS2
    return f"ex{label}" if has_ex and label != "unknown" else label


def validate_cn(code, ref_set):
    """Check code against the right HS-vintage reference set."""
    _, d, n = parse_code(code)
    if   n <= 4: return any(r.startswith(d) for r in ref_set)
    elif n == 6: return d in ref_set
    elif n >= 8: return d[:6] in ref_set
    return False


def get_digits(code):
    """Pure digits, ignoring ex prefix and spaces."""
    return re.sub(r"\s+", "", EX_RX.sub("", code))


def is_catch_all(code):
    """True for EU CN catch-all constructions:
       - HS4 chapter-catch-all: NN00 (e.g. ex9700 = ex chapter 97)
       - HS6+ heading/chapter-catch-all: NNNN 00, NNNN 00 00, NNNN 00 00 00
    """
    digits = get_digits(code)
    if len(digits) == 4 and digits[2:] == "00":
        return True
    return len(digits) >= 6 and digits[4:].rstrip("0") == ""


def detect_aggregation(cn_code, cn_type, all_codes):
    """If this code aggregates more specific codes in the same version, list them."""
    if cn_type in ("HS10", "exHS10"):
        return ""
    has_ex, d, _ = parse_code(cn_code)
    children = sorted(
        o for o in all_codes
        if o != cn_code
        and get_digits(o).startswith(d)
        and len(get_digits(o)) > len(d)
    )
    if not children:
        return ""
    note = f"aggregates: {', '.join(children)}"
    return f"[ex code] {note}" if has_ex else note


def split_multi_codes(cn_code, description, annex):
    """Split 'A or B, C' cells into one record each."""
    parts = [p.strip() for p in re.split(r",|\bor\b", cn_code) if p.strip()]
    return [{"annex": annex, "cn_code": p, "description": description}
            for p in parts]


def format_date_yyyymmdd(date_ddmmyyyy):
    parts = date_ddmmyyyy.split("-")
    return f"{parts[2]}-{parts[1]}-{parts[0]}" if len(parts) == 3 else date_ddmmyyyy


# Constants used by date_from_celex below.
ORIGINAL_ACT_ID = "32014R0833"
ORIGINAL_DATE   = "2014-07-31"

def date_from_celex(celex_id, soup=None):
    """Return YYYY-MM-DD for a CELEX ID.

    Three-tier strategy (preserves the historical behaviour):
      1. Body-text date (e.g. "31.7.2014") if `soup` is supplied — the most
         robust source for consolidated versions.
      2. Trailing YYYYMMDD on the CELEX ID — used by consolidated forms like
         02014R0833-20220316.
      3. Original-act fallback: 32014R0833 has no trailing date — hard-code
         the enactment date 2014-07-31.
    """
    if soup is not None:
        m = re.search(r"(\d{1,2})\.(\d{1,2})\.(\d{4})", soup.get_text())
        if m:
            date_ddmmyyyy = f"{m.group(1).zfill(2)}-{m.group(2).zfill(2)}-{m.group(3)}"
            return format_date_yyyymmdd(date_ddmmyyyy)
    m2 = re.search(r"(\d{4})(\d{2})(\d{2})$", celex_id)
    if m2:
        return f"{m2.group(1)}-{m2.group(2)}-{m2.group(3)}"
    if celex_id == ORIGINAL_ACT_ID:
        return ORIGINAL_DATE
    return celex_id   # surfaces as malformed downstream


# ============================================================
# SECTION 3 — EXTRACTION
# ============================================================

# =============================================================================
# PATCHED extract_cn_codes  —  RU CN extractor v6
# =============================================================================
# DROP-IN REPLACEMENT for the function of the same name in Section 3 of
# RU_CLEAN_CN_extraction_v5.ipynb. Nothing else in the notebook changes.
# (Bump the notebook to v6 on save; the helper functions parse_code,
# normalize_cn, classify_cn, validate_cn, detect_aggregation, split_multi_codes
# are unchanged and must remain in Section 2.)
#
# BUG FIXED (B.x — empty-leading-cell rows dropped in the 3-column template)
# -------------------------------------------------------------------------
# EUR-Lex OJ tables that mark the "ex" prefix in its own leading cell render a
# NON-ex code as an EMPTY leading cell with the code in cells[1]:
#       ["ex", "7113",     "Jewellery ..."]   <- ex code   (was handled)
#       ["",   "7102 10",  "Unsorted diamonds"] <- non-ex   (was DROPPED)
# The previous version had a branch only for first == "ex"; the empty-lead row
# then fell through to the standard 2-column branch and was swallowed by the
# `first.lower() in (..., "")` skip. Net effect: every non-ex code in such a
# table was silently lost.
#
# CONFIRMED IMPACT on the 34-version RU corpus (verified by re-extraction diff):
#   - Annex XXXVIIIA: recovers Part A (710210, 710231, 710239) and Part B
#     (710421, 710491) — the raw/synthetic diamond codes, i.e. the core of the
#     Art. 3p diamond IMPORT ban, previously absent from the panel.
#   - Annexes that are genuinely all-ex (XVIII, X post-2022-04-13, XXVI, XXVII)
#     are UNAFFECTED — the bug only bites a table that MIXES ex and non-ex rows.
#   - Plain 2-column annexes are UNAFFECTED.
#
# THE FIX: generalise the 3-column branch to handle both the ex case
# (first == "ex") and the non-ex case (first == "", code in cells[1]), guarded
# by a code-shape test on cells[1] so header rows ("CN code | Description") and
# note rows never match.
# =============================================================================

import re
import pandas as pd


# B.13: a code cell is a >=4-digit heading/subheading OR a bare 2-digit chapter
# (exactly two digits, optionally ex-prefixed). 1- and 3-digit tokens and
# numbered-note rows ("12.") stay rejected.
_HEADING_RX = re.compile(r"^(ex\s*)?\d{4}", re.IGNORECASE)        # B.13: case-insensitive (capital "Ex")
_CHAPTER_RX = re.compile(r"^(ex\s*)?\d{2}\s*$", re.IGNORECASE)     # B.13: case-insensitive (capital "Ex")

def _is_cn_code(s):
    return bool(_HEADING_RX.match(s) or _CHAPTER_RX.match(s))


def extract_cn_codes(soup, ref_set):
    """
    Parse HTML and return DataFrame of CN codes per Annex.
    Method A (original act): <p class='oj-tbl-cod'> + <p class='oj-tbl-txt'>
    Method B (consolidated): plain <td> cells inside <tr>
    B.12 (empty-lead-cell 3-column template) + B.13 (bare 2-digit chapters).
    """
    records       = []
    current_annex = "UNKNOWN"

    for tag in soup.find_all(True):

        # Annex titles
        cls = tag.get("class") or []
        if "oj-doc-ti" in cls or "title-annex-1" in cls:
            text = tag.get_text(strip=True).replace("\xa0", " ")
            if text.upper().startswith("ANNEX"):
                current_annex = text

        if tag.name != "tr":
            continue

        # ── Method A ──────────────────────────────────────────────────────
        cod_cells = tag.find_all("p", class_="oj-tbl-cod")
        if cod_cells:
            cn   = cod_cells[0].get_text(strip=True).replace("\xa0", " ")
            desc_cell = tag.find("p", class_="oj-tbl-txt")
            if desc_cell:
                desc = desc_cell.get_text(strip=True).replace("\xa0", " ")
            elif len(cod_cells) > 1:
                desc = cod_cells[1].get_text(strip=True).replace("\xa0", " ")
            else:
                desc = ""
            if _is_cn_code(cn):                                    # B.13
                records.extend(split_multi_codes(cn, desc, current_annex))
            continue

        # ── Method B ──────────────────────────────────────────────────────
        cells = tag.find_all("td")
        if len(cells) < 2:
            continue
        first  = cells[0].get_text(strip=True).replace("\xa0", " ")
        second = cells[1].get_text(strip=True).replace("\xa0", " ")

        # ── 3-column structure: <ex-or-empty leading cell> | code | desc ──
        # PATCH: the leading cell carries the "ex" marker (or is EMPTY for a
        # non-ex code). The code lives in cells[1]; the description in cells[2].
        # Handling first == "" here is the fix — previously those rows were
        # dropped by the "" entry in the standard-branch skip list below.
        # The re.match guard on `second` ensures header rows
        # (["", "CN code", "Description"]) and note rows never enter this branch.
        if (
            len(cells) >= 3
            and (first.lower() == "ex" or first == "")
            and _is_cn_code(second)                               # B.13
        ):
            is_ex     = first.lower() == "ex"
            desc_cell = cells[2].get_text(strip=True).replace("\xa0", " ")
            for part in re.split(r",|\bor\b", second):
                part = part.strip()
                if part and _is_cn_code(part):                    # B.13
                    if part.lower().startswith("ex"):
                        code = part                       # already ex-prefixed
                    elif is_ex:
                        code = f"ex{part}"                 # ex marker in cell 0
                    else:
                        code = part                        # genuine non-ex code
                    records.append({
                        "annex":       current_annex,
                        "cn_code":     code,
                        "description": desc_cell,
                    })
            continue

        # ── Standard 2-column structure ───────────────────────────────────
        if first.lower() in ("cn code", "cn codes", "cn  code", ""):
            continue
        if not _is_cn_code(first):                                # B.13
            continue
        records.extend(split_multi_codes(first, second, current_annex))

    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)
    df["cn_code"]      = df["cn_code"].apply(normalize_cn)
    df["cn_type"]      = df["cn_code"].apply(classify_cn)
    df["is_duplicate"] = df.duplicated(subset=["cn_code"], keep="first")
    df["in_reference"] = df["cn_code"].apply(lambda c: validate_cn(c, ref_set))

    all_codes = df["cn_code"].tolist()
    df["aggregation_note"] = df.apply(
        lambda r: detect_aggregation(r["cn_code"], r["cn_type"], all_codes),
        axis=1,
    )
    return df[["annex", "cn_code", "cn_type", "is_duplicate",
               "in_reference", "aggregation_note", "description"]]


# ============================================================
# SECTION 4 — QUALITY CHECKS
# ============================================================

def run_quality_checks(df, celex_id, date_str, vintage):
    L = ["=" * 60, "QUALITY CHECK REPORT",
         f"CELEX ID         : {celex_id}",
         f"Version date     : {date_str}",
         f"HS ref. vintage  : Y{vintage}",
         f"Total rows       : {len(df)}"]

    if len(df) == 0:
        L.append("\nNo CN codes found in this version.")
        return "\n".join(L)

    L += [f"Unique rows      : {(~df['is_duplicate']).sum()}",
          f"Duplicate rows   : {df['is_duplicate'].sum()}",
          "=" * 60]

    checks = [
        ("\n1. Rows with missing description    : ", df[df["description"] == ""],    ["annex", "cn_code"]),
        ("\n2. Rows with unknown CN type        : ", df[df["cn_type"] == "unknown"], ["annex", "cn_code"]),
        ("\n3. Codes not in reference list      : ", df[~df["in_reference"]],        ["annex", "cn_code", "cn_type"]),
    ]
    for header, sub, cols in checks:
        L.append(f"{header}{len(sub)}")
        if len(sub):
            L.append(sub[cols].to_string())

    L.append("\n4. CN type breakdown:")
    L.append(df["cn_type"].value_counts().to_string())

    L.append("\n5. CN codes per Annex:")
    summary = (df.groupby("annex")
                 .agg(total      = ("cn_code", "count"),
                      duplicates = ("is_duplicate", "sum"),
                      not_in_ref = ("in_reference", lambda x: (~x).sum()))
                 .reset_index())
    L.append(summary.to_string())
    return "\n".join(L)


# ============================================================
# SECTION 5 — (intentionally empty; removed 2026-05-20)
# ============================================================
# The former cross_version_analysis function lived here. It built
# annex-stability / annex-changes / ex-transitions / ex-redundancy
# tables across consecutive versions, treating each CN code as
# having a single per-version state.
#
# That approach is incompatible with the direction-aware downstream
# view: the same CN code in two different annexes now represents
# two distinct legal regimes (e.g. export ban vs anti-circumvention),
# not a movement of a single regime. When the future
# "treatment-over-time by import vs export" analysis becomes
# concrete, write it fresh against the R Step 3 panel × Python
# parser join, where direction is a column.
#
# The four output files this function used to produce
# (RU-ANNEX_STABILITY, RU-ANNEX_CHANGES, RU-EX_TRANSITIONS,
# RU-EX_REDUNDANCY) are no longer written. Archive previous
# versions of those files if you want to keep them.


# ============================================================
# SECTION 6 — MAIN LOOP
# ============================================================

print("Starting extraction loop...\n")

summary_records = []
per_version_dfs = {}    # date_yyyymmdd → DataFrame, kept for the
                        # not-in-reference rollup in Section 7 (avoids
                        # re-reading CSVs from disk)

for i, celex_id in enumerate(celex_ids):
    print(f"[{i+1}/{len(celex_ids)}] {celex_id} ...", end=" ", flush=True)

    try:
        html = get_html_by_celex_id(celex_id)
        if not html:
            raise ValueError("Empty response")
    except Exception as e:
        print(f"FAILED ({e})")
        summary_records.append({
            "celex_id": celex_id, "version_date": "unknown",
            "reference_vintage": "n/a", "filename": "n/a",
            "total_rows": 0, "unique_rows": 0, "duplicate_rows": 0,
            "not_in_reference": 0, "annexes_found": 0,
            "cn_types": "n/a", "missing_desc": 0,
            "unknown_type_rows": 0, "status": f"FAILED: {e}"})
        continue

    soup = BeautifulSoup(html, "html.parser")

    date_yyyymmdd = date_from_celex(celex_id, soup)
    vintage       = pick_vintage(date_yyyymmdd)

    df = extract_cn_codes(soup, cn_ref_sets[vintage])
    n_total  = len(df)
    n_unique = int((~df["is_duplicate"]).sum()) if n_total else 0
    n_dupes  = int(df["is_duplicate"].sum())    if n_total else 0
    n_no_ref = int((~df["in_reference"]).sum()) if n_total else 0
    print(f"{n_total} rows ({n_unique} unique) — {date_yyyymmdd}")

    # Cross-version state accumulators (kept slim: only the per-version
    # DataFrames needed by the NOT-IN-REFERENCE rollup below).
    if n_total:
        per_version_dfs[date_yyyymmdd] = df

    # Save per-version CSV + quality report (RU- prefix)
    csv_filename = f"RU-CN_extraction_{date_yyyymmdd}.csv"
    csv_path     = f"{output_folder}/{csv_filename}"
    empty_cols   = ["annex", "cn_code", "cn_type", "is_duplicate",
                    "in_reference", "aggregation_note", "description"]
    (df if n_total else pd.DataFrame(columns=empty_cols)).to_csv(
        csv_path, index=False, encoding="utf-8-sig")

    with open(f"{output_folder}/RU-CN_quality_{date_yyyymmdd}.txt", "w",
              encoding="utf-8") as f:
        f.write(run_quality_checks(df, celex_id, date_yyyymmdd, vintage))

    summary_records.append({
        "celex_id":          celex_id,
        "version_date":      date_yyyymmdd,
        "reference_vintage": f"Y{vintage}",
        "filename":          csv_filename,
        "total_rows":        n_total,
        "unique_rows":       n_unique,
        "duplicate_rows":    n_dupes,
        "not_in_reference":  n_no_ref,
        "annexes_found":     df["annex"].nunique() if n_total else 0,
        "cn_types":          (", ".join(f"{k}:{v}" for k, v
                              in df["cn_type"].value_counts().items())
                              if n_total else "n/a"),
        "missing_desc":      int((df["description"] == "").sum()) if n_total else 0,
        "unknown_type_rows": int((df["cn_type"] == "unknown").sum()) if n_total else 0,
        "status":            "OK"})

    time.sleep(2)


# ============================================================
# SECTION 7 — RETAINED CROSS-VERSION ARTEFACTS
# ============================================================
# Two pure per-version statistics are retained (no cross-version
# movement analysis):
#   - RU-NOT_IN_REFERENCE.csv: codes the validator could not place
#     in the right HS vintage, with is_catch_all (distinguishes EU
#     CN catch-all forms like ex2106 00 00 from genuine missing
#     codes like deprecated 441294 / 441860) and n_versions_not_in_ref
#     (persistence).
#   - RU-EX_EVOLUTION.csv: per-version counts of each HS digit level
#     (HS2..HS10) and its ex variants. HS2/exHS2 only fire if the
#     regulation contains chapter-level codes (e.g. "88" Aircraft
#     added in the 24 April 2026 amendment to Annex XI).

print("\nBuilding cross-version artefacts...")

# ── RU-NOT_IN_REFERENCE.csv ───────────────────────────────────
not_in_ref_rows = []
for date, df in per_version_dfs.items():
    sub = df[~df["in_reference"]]
    for _, r in sub.iterrows():
        not_in_ref_rows.append({
            "version_date": date, "annex": r["annex"],
            "cn_code": r["cn_code"], "cn_type": r["cn_type"],
            "is_catch_all": is_catch_all(r["cn_code"]),
            "description": r["description"]})

df_not_in_ref = pd.DataFrame(not_in_ref_rows)
if len(df_not_in_ref):
    n_map = (df_not_in_ref.groupby("cn_code")["version_date"]
             .nunique().reset_index()
             .rename(columns={"version_date": "n_versions_not_in_ref"}))
    df_not_in_ref = (df_not_in_ref.merge(n_map, on="cn_code", how="left")
                     .sort_values(["n_versions_not_in_ref", "cn_code", "version_date"],
                                  ascending=[False, True, True])
                     .reset_index(drop=True))

df_not_in_ref.to_csv(f"{output_folder}/RU-NOT_IN_REFERENCE.csv",
                     index=False, encoding="utf-8-sig")

# ── RU-EX_EVOLUTION.csv ───────────────────────────────────────
# Counts over ALL rows (incl. duplicates) so they reconcile with the
# cn_types breakdown in the summary. HS2/exHS2 columns are present
# regardless of vintage — they just hold zeros for versions that
# don't use chapter-level codes.
ALL_LEVELS = ["HS2", "HS4", "HS6", "HS8", "HS10",
              "exHS2", "exHS4", "exHS6", "exHS8", "exHS10"]
EX_LEVELS  = [t for t in ALL_LEVELS if t.startswith("ex")]

ex_evo_rows = []
for date, df in per_version_dfs.items():
    if df.empty:
        row = {"version_date": date, "total_rows": 0,
               "n_ex": 0, "n_non_ex": 0, "share_ex_pct": 0.0}
        for lvl in ALL_LEVELS:
            row[f"n_{lvl}"] = 0
        ex_evo_rows.append(row)
        continue
    counts  = df["cn_type"].value_counts()
    n_total = len(df)
    n_ex    = int(counts.reindex(EX_LEVELS, fill_value=0).sum())
    row = {"version_date":  date,
           "total_rows":    n_total,
           "n_ex":          n_ex,
           "n_non_ex":      n_total - n_ex,
           "share_ex_pct":  round(100 * n_ex / n_total, 2)}
    for lvl in ALL_LEVELS:
        row[f"n_{lvl}"] = int(counts.get(lvl, 0))
    ex_evo_rows.append(row)
df_ex_evo = pd.DataFrame(ex_evo_rows)
df_ex_evo.to_csv(f"{output_folder}/RU-EX_EVOLUTION.csv",
                 index=False, encoding="utf-8-sig")

# ── Console summary ───────────────────────────────────────────
print(f"  RU-NOT_IN_REFERENCE.csv  ({len(df_not_in_ref)} rows)")
if len(df_not_in_ref):
    n_catch   = int(df_not_in_ref["is_catch_all"].sum())
    n_genuine = int((~df_not_in_ref["is_catch_all"]).sum())
    print(f"    catch-all rows         : {n_catch}")
    print(f"    genuine missing rows   : {n_genuine}")
    print(f"    unique codes (all)     : {df_not_in_ref['cn_code'].nunique()}")
    print(f"    unique codes (genuine) : "
          f"{df_not_in_ref[~df_not_in_ref['is_catch_all']]['cn_code'].nunique()}")
print(f"  RU-EX_EVOLUTION.csv      ({len(df_ex_evo)} rows)")


# ============================================================
# SECTION 8 — SUMMARY + ZIP + DOWNLOAD
# ============================================================

print("\n" + "=" * 60)
print("OVERALL SUMMARY")
print("=" * 60)

df_summary = pd.DataFrame(summary_records)
print(df_summary[["celex_id", "version_date", "total_rows",
                  "unique_rows", "annexes_found", "status"]].to_string())

summary_path = f"{output_folder}/RU-SUMMARY_run_{run_timestamp}.csv"
df_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
print(f"\nSaved: {summary_path}")

print("\nCreating zip file...")
zip_path = f"/content/RU-CN_results_{run_timestamp}"
shutil.make_archive(zip_path, "zip", output_folder)
print(f"Zip created: {zip_path}.zip")

print("Downloading...")
files.download(f"{zip_path}.zip")

print(f"\nAll done!\nFiles also saved in Colab at: {output_folder}")

CN Code Extractor — EU Regulation 833/2014 — All Versions

Output folder: /content/RU-CN_extraction_results/RU-CN_results_2026-06-15_14-55-20

CN reference codes loaded:
  Y2012: 5205 codes
  Y2017: 5387 codes
  Y2022: 5612 codes
Versions to process      : 34

Starting extraction loop...

[1/34] 32014R0833 ... 30 rows (30 unique) — 2014-07-31
[2/34] 02014R0833-20140912 ... 29 rows (29 unique) — 2014-09-12
[3/34] 02014R0833-20141206 ... 30 rows (30 unique) — 2014-12-06
[4/34] 02014R0833-20151009 ... 30 rows (30 unique) — 2015-10-09
[5/34] 02014R0833-20171202 ... 30 rows (30 unique) — 2017-12-02
[6/34] 02014R0833-20190709 ... 30 rows (30 unique) — 2019-07-09
[7/34] 02014R0833-20220228 ... 86 rows (42 unique) — 2022-02-28
[8/34] 02014R0833-20220316 ... 760 rows (689 unique) — 2022-03-16
[9/34] 02014R0833-20220413 ... 1485 rows (1395 unique) — 2022-04-13
[10/34] 02014R0833-20220604 ... 1493 rows (1403 unique) — 2022-06-04
[11/34] 02014R0833-20220722 ... 1506 rows (1434 unique) — 2022-07-22

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All done!
Files also saved in Colab at: /content/RU-CN_extraction_results/RU-CN_results_2026-06-15_14-55-20


2 scripts to classify the "ex" codes

To confront the description of "ex" codes with the full description, a file is needed:

GENERAL-HSCodeandDescription.xlsx

In [5]:
"""
============================================================
EX-EXCLUSION EXPLORATORY ANALYSIS — standalone script
============================================================

Purpose:
  Scan all 34 per-version CN extraction CSVs and, for each row whose
  cn_code starts with "ex", inspect the description for signs of an
  exclusion clause ("except for", "other than", "excluding", etc.).

  This is a DIAGNOSTIC tool, not a parser. It reports counts and
  produces two CSVs — one of confident hits, one of edge cases for
  human review — so we can decide whether a full parser is worth
  building.

Usage in Colab:

  By default this script auto-links to the folder produced by the main
  extractor (cell 3): the variable `output_folder` is reused if it
  exists in the kernel namespace. To override (e.g. to run against a
  previous run after restarting the kernel), set INPUT_FOLDER below
  to a fixed path like "/content/RU-CN_extraction_results/RU-CN_results_<timestamp>".

Outputs:
    RU-EX_EXCLUSIONS_SUMMARY.csv  one row per version with counts
    RU-EX_EXCLUSIONS_AUTO.csv     rows where keyword AND CN-code pattern
                                  were both found in the description
    RU-EX_EXCLUSIONS_REVIEW.csv   rows where one or the other was found
                                  but not both — candidates for human review

Approach (Strategy 2 — broad scan + manual review):
  - Cast a wide net on exclusion keywords (10+ phrasings)
  - Look separately for CN-code-shaped patterns
  - Auto-classify into three buckets per ex row:
      hit       : keyword found AND code(s) found
      review-A  : keyword found but no codes detected
      review-B  : codes detected in description but no keyword
                  (could be aggregation note rather than exclusion)
      none      : neither — most ex rows fall here
"""

import pandas as pd
import re
from pathlib import Path


# ============================================================
# CONFIGURE — set INPUT_FOLDER to override the auto-link
# ============================================================
# By default, reuse the folder created by cell 3 (the main extractor).
# To run standalone, replace `output_folder` below with a fixed path
# like "/content/RU-CN_extraction_results/RU-CN_results_<timestamp>".
try:
    INPUT_FOLDER = output_folder
except NameError:
    INPUT_FOLDER = "/content/RU-CN_extraction_results/RU-CN_results_<timestamp>"
# ============================================================


# ── Exclusion keywords (case-insensitive, broad net) ────────
EXCLUSION_PHRASES = [
    "except for",
    "except those",
    "except the",
    "with the exception of",
    "other than",
    "excluding",
    "to the exclusion of",
    "save for",
    "not including",
    "but not",        # very loose — will trigger review-A often, by design
]

EXCLUSION_RX = re.compile(
    r"\b(" + "|".join(re.escape(p) for p in EXCLUSION_PHRASES) + r")\b",
    re.IGNORECASE,
)

# Match CN code shapes inside a description.
# Accepts both spaced ("2835 26 00") and glued ("28352600") forms,
# plus mixed forms ("2835 2600"). Digit count must be exactly 4/6/8/10.
CODE_IN_DESC_RX = re.compile(
    r"\b("
    r"\d{2}\s?\d{2}"           # 4 digits, optional space between halves
    r"(?:\s?\d{2}){0,3}"       # up to 3 more 2-digit groups, each with optional preceding space
    r")\b"
)


def find_excluded_codes(description):
    """Return list of CN-code-shaped strings, normalised to 'XXXX XX...'."""
    if not isinstance(description, str):
        return []
    raw = CODE_IN_DESC_RX.findall(description)
    out = []
    for s in raw:
        digits = re.sub(r"\s+", "", s)
        if len(digits) < 4:
            continue
        if   len(digits) == 4: norm = digits
        elif len(digits) == 6: norm = f"{digits[:4]} {digits[4:]}"
        elif len(digits) == 8: norm = f"{digits[:4]} {digits[4:6]} {digits[6:]}"
        elif len(digits) == 10: norm = f"{digits[:4]} {digits[4:6]} {digits[6:8]} {digits[8:]}"
        else: continue
        out.append(norm)
    return out


def find_exclusion_keywords(description):
    """Return list of distinct exclusion phrases found in the description."""
    if not isinstance(description, str):
        return []
    matches = EXCLUSION_RX.findall(description)
    seen, out = set(), []
    for m in matches:
        ml = m.lower()
        if ml not in seen:
            seen.add(ml)
            out.append(ml)
    return out


# ============================================================
# MAIN
# ============================================================

folder = Path(INPUT_FOLDER)
if not folder.is_dir():
    raise FileNotFoundError(f"Folder not found: {folder}")

csv_files = sorted(folder.glob("RU-CN_extraction_*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No RU-CN_extraction_*.csv files found in {folder}")

print(f"Scanning {len(csv_files)} per-version CSVs in {folder}\n")

summary_rows  = []
auto_rows     = []
review_rows   = []

for path in csv_files:
    version_date = path.stem.replace("RU-CN_extraction_", "")

    df = pd.read_csv(path, dtype=str).fillna("")
    if len(df) == 0:
        summary_rows.append({
            "version_date": version_date,
            "n_rows": 0, "n_ex": 0,
            "n_hit": 0, "n_review_A": 0, "n_review_B": 0, "n_none": 0,
        })
        continue

    is_ex = df["cn_code"].str.match(r"^ex", case=False, na=False)
    ex_df = df[is_ex].copy()

    n_ex      = len(ex_df)
    n_hit     = 0
    n_rev_a   = 0
    n_rev_b   = 0

    for _, r in ex_df.iterrows():
        desc = r["description"]
        kws  = find_exclusion_keywords(desc)
        codes_in_desc = find_excluded_codes(desc)

        # Filter out the row's own parent code from the "exclusions"
        own_digits = re.sub(r"\s+", "",
                            re.sub(r"^ex\s*", "", r["cn_code"], flags=re.I))
        codes_filtered = []
        for c in codes_in_desc:
            c_digits = re.sub(r"\s+", "", c)
            if c_digits == own_digits or c_digits == own_digits[:len(c_digits)]:
                continue
            codes_filtered.append(c)

        has_kw   = bool(kws)
        has_code = bool(codes_filtered)

        record = {
            "version_date":      version_date,
            "annex":              r["annex"],
            "cn_code":            r["cn_code"],
            "exclusion_keywords": ", ".join(kws) if kws else "",
            "excluded_codes":     ", ".join(codes_filtered) if codes_filtered else "",
            "description":        desc,
        }

        if has_kw and has_code:
            n_hit += 1
            record["bucket"] = "hit"
            auto_rows.append(record)
        elif has_kw and not has_code:
            n_rev_a += 1
            record["bucket"] = "review-A (keyword, no codes)"
            review_rows.append(record)
        elif has_code and not has_kw:
            n_rev_b += 1
            record["bucket"] = "review-B (codes, no keyword)"
            review_rows.append(record)
        # else: skip — neither found

    n_none = n_ex - n_hit - n_rev_a - n_rev_b

    summary_rows.append({
        "version_date": version_date,
        "n_rows":       len(df),
        "n_ex":         n_ex,
        "n_hit":        n_hit,
        "n_review_A":   n_rev_a,
        "n_review_B":   n_rev_b,
        "n_none":       n_none,
    })


# ============================================================
# WRITE OUTPUTS
# ============================================================

df_summary = pd.DataFrame(summary_rows)
df_auto    = pd.DataFrame(auto_rows)
df_review  = pd.DataFrame(review_rows)

df_summary.to_csv(folder / "RU-EX_EXCLUSIONS_SUMMARY.csv",
                  index=False, encoding="utf-8-sig")
df_auto.to_csv(folder / "RU-EX_EXCLUSIONS_AUTO.csv",
               index=False, encoding="utf-8-sig")
df_review.to_csv(folder / "RU-EX_EXCLUSIONS_REVIEW.csv",
                 index=False, encoding="utf-8-sig")


# ============================================================
# REPORT
# ============================================================

print(df_summary.to_string(index=False))
print()
print("=" * 60)
print("TOTALS ACROSS ALL VERSIONS")
print("=" * 60)
totals = df_summary[["n_ex", "n_hit", "n_review_A", "n_review_B", "n_none"]].sum()
print(totals.to_string())
print()

if df_auto.empty:
    print("No confident hits found.")
else:
    pct_hit = 100 * len(df_auto) / totals["n_ex"]
    print(f"Confident hits (keyword + codes): {len(df_auto)}  ({pct_hit:.1f}% of all ex rows)")
    print()
    print("Frequency of ex_code rows in AUTO bucket (all unique codes):")
    print(df_auto.groupby("cn_code").size()
          .sort_values(ascending=False).to_string())

print()
print(f"Files written to: {folder}")
print(f"  RU-EX_EXCLUSIONS_SUMMARY.csv  ({len(df_summary)} rows)")
print(f"  RU-EX_EXCLUSIONS_AUTO.csv     ({len(df_auto)} rows)")
print(f"  RU-EX_EXCLUSIONS_REVIEW.csv   ({len(df_review)} rows)")

Scanning 34 per-version CSVs in /content/RU-CN_extraction_results/RU-CN_results_2026-06-15_14-55-20

version_date  n_rows  n_ex  n_hit  n_review_A  n_review_B  n_none
  2014-07-31      30     3      0           0           3       0
  2014-09-12      29     2      0           0           2       0
  2014-12-06      30     5      0           0           3       2
  2015-10-09      30     5      0           0           3       2
  2017-12-02      30     5      0           0           3       2
  2019-07-09      30     5      0           0           3       2
  2022-02-28      86     5      0           0           3       2
  2022-03-16     760   341     14          14          13     300
  2022-04-13    1485   405     14          19          13     359
  2022-06-04    1493   410     19          19          13     359
  2022-07-22    1506   395     19          19          13     344
  2022-10-07    1720   392     14          19          13     346
  2022-12-04    1721   392     14        

In [6]:
"""
============================================================
EX-EXCLUSION DEDUPLICATION — annotation-ready spreadsheet
============================================================

Purpose:
  Take the three CSVs produced by ex_exclusions_explorer.py
  (RU-EX_EXCLUSIONS_AUTO.csv and RU-EX_EXCLUSIONS_REVIEW.csv)
  and merge them into a single deduplicated Excel workbook,
  ready for manual annotation.

Usage in Colab:

  By default this script auto-links to the folder produced by the main
  extractor (cell 3): the variable `output_folder` is reused if it
  exists in the kernel namespace. To override, set INPUT_FOLDER below
  to a fixed path. The HS reference file (GENERAL-HSCodeandDescription.xlsx)
  must be uploaded to /content/ separately.

Output:
    RU-EX_EXCLUSIONS_FOR_REVIEW.xlsx
        Sheet 1 — 'exclusions'
            One row per unique (bucket, cn_code, annex, description).
            Sorted by bucket (hit / review-A / review-B), then by
            descending n_versions.
        Sheet 2 — 'unique_codes'
            One row per unique cn_code. Most aggressive deduplication:
            collapses across annexes, descriptions, and buckets.
            Sorted by descending n_versions.

  Both sheets have empty 'manual_excluded_codes' and 'manual_notes'
  columns at the right edge for human annotation.
"""

import pandas as pd
import re
from pathlib import Path


# ============================================================
# CONFIGURE — set INPUT_FOLDER to override the auto-link
# ============================================================
# By default, reuse the folder created by cell 3 (the main extractor).
# To run standalone, replace `output_folder` below with a fixed path
# like "/content/RU-CN_extraction_results/RU-CN_results_<timestamp>".
try:
    INPUT_FOLDER = output_folder
except NameError:
    INPUT_FOLDER = "/content/RU-CN_extraction_results/RU-CN_results_<timestamp>"

HS_REFERENCE_FILE = "/content/GENERAL-HSCodeandDescription.xlsx"
# ============================================================


folder = Path(INPUT_FOLDER)
if not folder.is_dir():
    raise FileNotFoundError(f"Folder not found: {folder}")

auto_path   = folder / "RU-EX_EXCLUSIONS_AUTO.csv"
review_path = folder / "RU-EX_EXCLUSIONS_REVIEW.csv"

if not auto_path.exists() or not review_path.exists():
    raise FileNotFoundError(
        f"Expected RU-EX_EXCLUSIONS_AUTO.csv and RU-EX_EXCLUSIONS_REVIEW.csv\n"
        f"in {folder}. Run ex_exclusions_explorer.py first.")

def _safe_read(path):
    """Read CSV; return empty DataFrame with default columns if file has no rows."""
    try:
        return pd.read_csv(path, dtype=str).fillna("")
    except pd.errors.EmptyDataError:
        return pd.DataFrame(columns=[
            "version_date", "annex", "cn_code", "exclusion_keywords",
            "excluded_codes", "description", "bucket"])

# ── Load and stack the three buckets ─────────────────────────
df_auto   = _safe_read(auto_path)
df_review = _safe_read(review_path)
df_all    = pd.concat([df_auto, df_review], ignore_index=True)

print(f"Loaded {len(df_auto)} AUTO rows + {len(df_review)} REVIEW rows "
      f"= {len(df_all)} total occurrences")


# ── Load HS reference descriptions (3 vintages) ──────────────
hs_path = Path(HS_REFERENCE_FILE)
if not hs_path.exists():
    raise FileNotFoundError(
        f"HS reference file not found: {hs_path}\n"
        f"Adjust HS_REFERENCE_FILE at the top of this script.")

def _load_hs_sheet(sheet):
    df_hs = pd.read_excel(hs_path, sheet_name=sheet, dtype=str)
    df_hs = df_hs[["Code", "Description"]].dropna(subset=["Code"])
    df_hs["Code"] = df_hs["Code"].str.strip()
    df_hs = df_hs[df_hs["Code"].str.match(r"^\d+$")]
    return dict(zip(df_hs["Code"], df_hs["Description"]))

hs_refs = {
    "2012": _load_hs_sheet("HS12"),
    "2017": _load_hs_sheet("HS17"),
    "2022": _load_hs_sheet("HS22"),
}
print(f"Loaded HS reference descriptions:")
for v, d in hs_refs.items():
    print(f"  HS{v}: {len(d)} entries")
print()


def pick_vintage(date_yyyymmdd):
    """Same cliff-edges as the main extractor — picks HS revision in force."""
    if not isinstance(date_yyyymmdd, str) or len(date_yyyymmdd) < 10:
        return "2022"
    if date_yyyymmdd < "2017-01-01": return "2012"
    if date_yyyymmdd < "2022-01-01": return "2017"
    return "2022"


def lookup_full_description(cn_code, vintage):
    """
    Look up the official HS description for an ex-prefixed code.
    Returns (description, match_level) where match_level is one of:
      'exact'      — found at the exact digit length
      'hs6'        — fell back to the 6-digit parent
      'hs4'        — fell back to the 4-digit heading
      'not_found'  — not found at any level
    """
    ref = hs_refs.get(vintage, hs_refs["2022"])
    digits = re.sub(r"\s+", "",
                    re.sub(r"^ex\s*", "", str(cn_code), flags=re.IGNORECASE))
    if not digits.isdigit():
        return ("", "not_found")
    if digits in ref:                         return (ref[digits], "exact")
    if len(digits) > 6 and digits[:6] in ref: return (ref[digits[:6]], "hs6")
    if len(digits) > 4 and digits[:4] in ref: return (ref[digits[:4]], "hs4")
    if len(digits) == 4 and digits in ref:    return (ref[digits], "exact")
    return ("", "not_found")


# ── Deduplicate by (bucket, cn_code, annex, description) ─────

# ── Build sheet 1: 'exclusions' ──────────────────────────────
# Group by (bucket, cn_code, annex, description) — preserves all context.
group_cols = ["bucket", "cn_code", "annex", "description"]

agg = (
    df_all
    .groupby(group_cols, sort=False)
    .agg(
        n_versions          = ("version_date",        "nunique"),
        first_version       = ("version_date",        "min"),
        last_version        = ("version_date",        "max"),
        exclusion_keywords  = ("exclusion_keywords",  "first"),
        excluded_codes      = ("excluded_codes",      "first"),
    )
    .reset_index()
)

bucket_order = {"hit": 0,
                "review-A (keyword, no codes)": 1,
                "review-B (codes, no keyword)": 2}
agg["_bucket_order"] = agg["bucket"].map(bucket_order).fillna(99)
agg = (agg
       .sort_values(["_bucket_order", "n_versions", "cn_code"],
                    ascending=[True, False, True])
       .drop(columns="_bucket_order")
       .reset_index(drop=True))

# ── Add empty columns for manual annotation ──────────────────
agg["manual_excluded_codes"] = ""
agg["manual_notes"]          = ""

# ── Add full HS description columns (per-row vintage from first_version) ─
def _add_full_desc(row):
    desc, level = lookup_full_description(row["cn_code"],
                                          pick_vintage(row["first_version"]))
    return pd.Series({"full_code_description": desc,
                      "full_code_match_level": level})

agg = pd.concat([agg, agg.apply(_add_full_desc, axis=1)], axis=1)

# Final column order — full_code_description sits next to description
agg = agg[[
    "bucket", "cn_code", "annex",
    "n_versions", "first_version", "last_version",
    "exclusion_keywords", "excluded_codes",
    "description", "full_code_description", "full_code_match_level",
    "manual_excluded_codes", "manual_notes",
]]

# ── Build sheet 2: 'unique_codes' ────────────────────────────
# Groups by cn_code alone, collapsing all annexes/descriptions/buckets
# for the same code into one row. The most aggressive deduplication.
unique_codes = (
    df_all
    .groupby("cn_code", sort=False)
    .agg(
        n_occurrences      = ("version_date",  "count"),
        n_versions         = ("version_date",  "nunique"),
        first_version      = ("version_date",  "min"),
        last_version       = ("version_date",  "max"),
        annexes            = ("annex",         lambda x: ", ".join(sorted(set(x)))),
        n_annexes          = ("annex",         "nunique"),
        buckets            = ("bucket",        lambda x: ", ".join(sorted(set(x)))),
        n_descriptions     = ("description",   "nunique"),
        sample_description = ("description",   "first"),
    )
    .reset_index()
    .sort_values(["n_versions", "cn_code"], ascending=[False, True])
    .reset_index(drop=True)
)
unique_codes["manual_excluded_codes"] = ""
unique_codes["manual_notes"]          = ""

# Add full HS description (per-row vintage from first_version)
unique_codes = pd.concat([unique_codes,
                          unique_codes.apply(_add_full_desc, axis=1)],
                         axis=1)

unique_codes = unique_codes[[
    "cn_code", "n_occurrences", "n_versions", "n_annexes", "n_descriptions",
    "first_version", "last_version", "annexes", "buckets",
    "sample_description", "full_code_description", "full_code_match_level",
    "manual_excluded_codes", "manual_notes",
]]

# ── Write Excel with light formatting ────────────────────────
out_path = folder / "RU-EX_EXCLUSIONS_FOR_REVIEW.xlsx"

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    agg.to_excel(writer, sheet_name="exclusions",
                 index=False, freeze_panes=(1, 0))
    unique_codes.to_excel(writer, sheet_name="unique_codes",
                          index=False, freeze_panes=(1, 0))

    from openpyxl.styles import Alignment
    from openpyxl.utils import get_column_letter

    # Columns whose name suggests long text — always wrap them, even if
    # currently empty (e.g. manual annotation columns waiting to be filled).
    WRAP_NAMES = {"description", "full_code_description",
                  "sample_description", "notes", "manual_notes",
                  "manual_excluded_codes"}

    def _autofit(ws, df, max_width=80, wrap_threshold=40):
        """
        Set column widths from content. Columns whose computed width
        hits wrap_threshold OR whose name is in WRAP_NAMES get text
        wrapping enabled. Width = longest of (header, any cell value),
        padded by 2, capped at max_width.
        """
        for i, col in enumerate(df.columns, start=1):
            longest = max(
                [len(str(col))]
                + [len(str(v)) for v in df[col].astype(str).tolist()]
            )
            width = min(longest + 2, max_width)
            letter = get_column_letter(i)
            ws.column_dimensions[letter].width = width

            should_wrap = (
                width >= wrap_threshold
                or col.lower() in WRAP_NAMES
            )
            if should_wrap:
                for cell in ws[letter]:
                    cell.alignment = Alignment(wrap_text=True, vertical="top")

    _autofit(writer.sheets["exclusions"],   agg)
    _autofit(writer.sheets["unique_codes"], unique_codes)

# ── Report ───────────────────────────────────────────────────
print()
print(f"Sheet 'exclusions'    : {len(agg)} unique (bucket, cn_code, annex, description) rows")
print(f"Sheet 'unique_codes'  : {len(unique_codes)} unique cn_codes")
print()
print("Breakdown by bucket (sheet 'exclusions'):")
print(agg["bucket"].value_counts().to_string())
print()
print("Breakdown of full_code_match_level (sheet 'exclusions'):")
print(agg["full_code_match_level"].value_counts().to_string())
print()
print(f"File written: {out_path}")
print()
print("Full list of exclusion rules (sheet 'exclusions'):")
print(agg[["bucket", "cn_code", "annex", "n_versions",
           "exclusion_keywords"]].to_string(index=False))
print()
print("Full list of unique cn_codes (sheet 'unique_codes'):")
print(unique_codes[["cn_code", "n_versions", "n_annexes",
                    "n_descriptions"]].to_string(index=False))

# ── Auto-download in Colab ───────────────────────────────────
# google.colab is only available in the Colab runtime; if running
# locally just print a hint and skip silently.
try:
    from google.colab import files
    print()
    print("Downloading file...")
    files.download(str(out_path))
except ImportError:
    print()
    print(f"(Not running in Colab — file is at {out_path})")

Loaded 459 AUTO rows + 949 REVIEW rows = 1408 total occurrences
Loaded HS reference descriptions:
  HS2012: 6528 entries
  HS2017: 6708 entries
  HS2022: 6939 entries


Sheet 'exclusions'    : 110 unique (bucket, cn_code, annex, description) rows
Sheet 'unique_codes'  : 70 unique cn_codes

Breakdown by bucket (sheet 'exclusions'):
bucket
hit                             48
review-A (keyword, no codes)    37
review-B (codes, no keyword)    25

Breakdown of full_code_match_level (sheet 'exclusions'):
full_code_match_level
exact    46
hs4      37
hs6      27

File written: /content/RU-CN_extraction_results/RU-CN_results_2026-06-15_14-55-20/RU-EX_EXCLUSIONS_FOR_REVIEW.xlsx

Full list of exclusion rules (sheet 'exclusions'):
                      bucket      cn_code          annex  n_versions     exclusion_keywords
                         hit ex6217 00 00    ANNEX XVIII          27             other than
                         hit ex8603 00 00    ANNEX XVIII          27             other 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Script to enrich the "summary" csv with graph-ready summaries

In [7]:
# =============================================================================
# CELL — enrich CN-extractor output with graph-ready summaries        [RU]
# Paste as a NEW cell and run AFTER the extraction loop. It reads the
# per-version RU-CN_extraction_*.csv the extractor just wrote and writes two
# tidy, chart-ready CSVs into the same results folder (timestamped, so reruns
# never overwrite). No re-extraction needed.
#
#   RU-CN_summary_by_version_<ts>.csv   one row per version (quick time series)
#       version, celex_id, hs_vintage, n_rows, n_unique, n_dup, n_annexes,
#       n_distinct_codes, n_chapter_codes(=lvl2), lvl2_n..lvl10_n,
#       ex_n, plain_n, not_in_ref_n, and per-type HS2_n..exHS8_n
#
#   RU-CN_summary_long_<ts>.csv         one row per (version, annex, cn_type)
#       version, annex, cn_type, level, is_ex, n_codes, n_rows, n_not_in_ref
#
# COUNTING:
#   LONG (per-annex view): n_codes = distinct cn_code WITHIN that annex (a code
#     in two annexes appears in both — true per-annex coverage). n_rows = all.
#   WIDE (per-version view): n_unique / lvl*_n / type counts = distinct codes in
#     the version, DEDUPED across annexes (is_duplicate == False).
#   "level" = digit granularity the LAW used (2 = bare chapter, 4 = heading,
#     6 = HS6, 8/10 = CN8/CN10); ex/plain folded in level counts, split in
#     is_ex / the type columns.
# -----------------------------------------------------------------------------
# This cell reuses the notebook's own `output_folder` (the RU-CN_results_<ts>/
# the extractor just wrote) — so run it right after the extraction loop and it
# needs NO configuration. The two settings below are only fallbacks for running
# it later in a fresh session:
#   RESULTS_DIR : set to a specific results folder to force it, else None.
#   BASE_DIR    : parent searched (recursively) for RU-CN_results_* when
#                 `output_folder` is not in memory.
RESULTS_DIR = None
BASE_DIR    = "/content/RU-CN_extraction_results"
# =============================================================================
import os, re, glob
from datetime import datetime
import pandas as pd

if RESULTS_DIR is None:
    RESULTS_DIR = globals().get("output_folder", None)   # set by the extractor run
if RESULTS_DIR is None:
    cands = [d for d in glob.glob(os.path.join(BASE_DIR, "**", "RU-CN_results_*"),
                                  recursive=True) if os.path.isdir(d)]
    if not cands and glob.glob(os.path.join(BASE_DIR, "RU-CN_extraction_*.csv")):
        cands = [BASE_DIR]                                # CSVs sit directly in BASE_DIR
    if not cands:
        found = glob.glob(os.path.join(BASE_DIR, "**", "RU-CN_extraction_*.csv"),
                          recursive=True)
        raise RuntimeError(
            "Could not locate the extractor results. Run this right after the "
            "extraction loop (so `output_folder` is in memory), or set RESULTS_DIR "
            f"to the RU-CN_results_* folder.\nSearched BASE_DIR={BASE_DIR!r}; "
            f"extraction CSVs found there: {len(found)}.")
    RESULTS_DIR = max(cands, key=os.path.getmtime)
print("results folder:", RESULTS_DIR)

RUN_TS     = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
DUP_TRUE   = {"True", "TRUE", "true", "1"}
TYPE_ORDER = ["HS2", "exHS2", "HS4", "exHS4", "HS6", "exHS6",
              "HS8", "exHS8", "HS10", "exHS10"]


def quality_meta(folder, date_str):
    """celex_id + hs_vintage from the matching quality .txt, if present."""
    p = os.path.join(folder, f"RU-CN_quality_{date_str}.txt")
    celex = vint = ""
    if os.path.exists(p):
        txt = open(p, encoding="utf-8", errors="ignore").read()
        m = re.search(r"CELEX ID\s*:\s*(\S+)", txt);        celex = m.group(1) if m else ""
        m = re.search(r"HS ref\. vintage\s*:\s*(\S+)", txt); vint  = m.group(1) if m else ""
    return celex, vint


def load_version(path):
    date_str = re.search(r"(\d{4}-\d{2}-\d{2})", os.path.basename(path)).group(1)
    d = pd.read_csv(path, dtype=str).fillna("")
    d["version"] = date_str
    d["is_dup"]  = d["is_duplicate"].isin(DUP_TRUE)
    d["level"]   = d["cn_type"].str.extract(r"(\d+)$").astype("Int64")
    d["is_ex"]   = d["cn_type"].str.startswith("ex")
    d["in_ref"]  = d["in_reference"].isin(DUP_TRUE)
    return d, date_str


files = sorted(glob.glob(os.path.join(RESULTS_DIR, "RU-CN_extraction_*.csv")))
if not files:
    raise RuntimeError(f"No RU-CN_extraction_*.csv in {RESULTS_DIR}")

long_rows, wide_rows = [], []
for f in files:
    d, date_str = load_version(f)
    u = d[~d["is_dup"]]
    celex, vint = quality_meta(RESULTS_DIR, date_str)

    g = (d.groupby(["annex", "cn_type"])
           .apply(lambda x: pd.Series({
               "n_rows":       len(x),
               "n_codes":      x["cn_code"].nunique(),
               "n_not_in_ref": x.loc[~x["in_ref"], "cn_code"].nunique(),
           }), include_groups=False)
           .reset_index())
    g.insert(0, "version", date_str)
    g["level"] = g["cn_type"].str.extract(r"(\d+)$").astype(int)
    g["is_ex"] = g["cn_type"].str.startswith("ex")
    long_rows.append(g[["version", "annex", "cn_type", "level", "is_ex",
                        "n_codes", "n_rows", "n_not_in_ref"]])

    row = {"version": date_str, "celex_id": celex, "hs_vintage": vint,
           "n_rows": len(d), "n_unique": len(u), "n_dup": int(d["is_dup"].sum()),
           "n_annexes": d["annex"].nunique(),
           "n_distinct_codes": u["cn_code"].nunique(),
           "n_chapter_codes": int((u["level"] == 2).sum()),
           "ex_n": int(u["is_ex"].sum()), "plain_n": int((~u["is_ex"]).sum()),
           "not_in_ref_n": int((~u["in_ref"]).sum())}
    for lvl in [2, 4, 6, 8, 10]:
        row[f"lvl{lvl}_n"] = int((u["level"] == lvl).sum())
    vc = u["cn_type"].value_counts()
    for t in TYPE_ORDER:
        row[f"{t}_n"] = int(vc.get(t, 0))
    wide_rows.append(row)

long_df = pd.concat(long_rows, ignore_index=True).sort_values(
    ["version", "annex", "level", "cn_type"])
wide_df = pd.DataFrame(wide_rows).sort_values("version")

# carry over `status` from the extractor's own RU-SUMMARY_run, if present,
# so the enriched by-version file is a clean superset of it
_runs = sorted(glob.glob(os.path.join(RESULTS_DIR, "RU-SUMMARY_run_*.csv")))
if _runs:
    _base = pd.read_csv(_runs[-1], dtype=str)
    if {"version_date", "status"} <= set(_base.columns):
        wide_df = (wide_df.merge(_base[["version_date", "status"]],
                                 left_on="version", right_on="version_date", how="left")
                          .drop(columns=["version_date"]))

out_long = os.path.join(RESULTS_DIR, f"RU-CN_summary_long_{RUN_TS}.csv")
out_wide = os.path.join(RESULTS_DIR, f"RU-CN_summary_by_version_{RUN_TS}.csv")
long_df.to_csv(out_long, index=False)
wide_df.to_csv(out_wide, index=False)

print(f"versions processed : {len(files)}")
print(f"long  rows         : {len(long_df)}  -> {os.path.basename(out_long)}")
print(f"wide  rows         : {len(wide_df)}  -> {os.path.basename(out_wide)}")
print("\nby-version preview:")
_cols = ["version", "n_unique", "n_chapter_codes",
         "lvl2_n", "lvl4_n", "lvl6_n", "lvl8_n", "ex_n", "not_in_ref_n"]
print(wide_df[_cols].to_string(index=False))

results folder: /content/RU-CN_extraction_results/RU-CN_results_2026-06-15_14-55-20
versions processed : 34
long  rows         : 1190  -> RU-CN_summary_long_2026-06-15_15-00-33.csv
wide  rows         : 34  -> RU-CN_summary_by_version_2026-06-15_15-00-33.csv

by-version preview:
   version  n_unique  n_chapter_codes  lvl2_n  lvl4_n  lvl6_n  lvl8_n  ex_n  not_in_ref_n
2014-07-31        30                0       0       0       5      25     3             0
2014-09-12        29                0       0       0       4      25     2             0
2014-12-06        30                0       0       0       5      25     5             0
2015-10-09        30                0       0       0       5      25     5             0
2017-12-02        30                0       0       0       5      25     5             0
2019-07-09        30                0       0       0       5      25     5             0
2022-02-28        42                1       1       0       5      36     5             0
2

Script to check differences between different runs of the script

In [ ]:
"""
============================================================
DIFF SCRIPT — compare outputs of the V1 vs V2 versions
============================================================

Usage in Colab (or locally):

  Step 1 — run the V1 version of the extractor.
           In Colab, find the resulting folder
              /content/RU-CN_extraction_results/RU-CN_results_<timestamp_v1>
           Either keep it as-is or rename it to something like
              /content/RU-CN_results_V1

  Step 2 — run the V2 version of the extractor.
           Same idea:
              /content/RU-CN_extraction_results/RU-CN_results_<timestamp_v2>
           or rename to
              /content/RU-CN_results_V2

  Step 3 — set FOLDER_V1 and FOLDER_V2 below to the
           two paths, then run this script.

The script compares every file with the same NAME in both
folders. CSVs are compared row-by-row (after sorting, so row
order doesn't matter). Text files are compared line-by-line.
Files that exist in only one of the two folders are flagged.

Final verdict: "ALL FILES IDENTICAL" or a detailed report.

Note: the RU-SUMMARY_run_<timestamp>.csv file has a timestamp
in its name, so it appears under different filenames in the
two folders. The script handles this automatically by matching
any file starting with "RU-SUMMARY_run_".
"""

import os
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIGURE THESE TWO PATHS
# ============================================================
FOLDER_V1    = "/content/RU-CN_results_V1"
FOLDER_V2 = "/content/RU-CN_results_V2"
# ============================================================


def list_files(folder):
    """Return {filename: full_path} for all files directly in folder."""
    p = Path(folder)
    if not p.is_dir():
        raise FileNotFoundError(f"Folder not found: {folder}")
    return {f.name: f for f in p.iterdir() if f.is_file()}


def normalise_summary_name(name):
    """RU-SUMMARY_run_<timestamp>.csv → RU-SUMMARY_run_*.csv (for matching)."""
    if name.startswith("RU-SUMMARY_run_") and name.endswith(".csv"):
        return "RU-SUMMARY_run_*.csv"
    return name


def compare_csv(path_a, path_b):
    """
    Row-by-row comparison ignoring row order.
    Returns (is_identical, message).
    """
    df_a = pd.read_csv(path_a, dtype=str).fillna("")
    df_b = pd.read_csv(path_b, dtype=str).fillna("")

    # 1. Column comparison
    if list(df_a.columns) != list(df_b.columns):
        return False, (f"COLUMN MISMATCH\n"
                       f"  v1   : {list(df_a.columns)}\n"
                       f"  v2: {list(df_b.columns)}")

    # 2. Row count
    if len(df_a) != len(df_b):
        return False, f"ROW COUNT: v1={len(df_a)} v2={len(df_b)}"

    # 3. Row-by-row, sorted (so order doesn't matter)
    cols   = list(df_a.columns)
    df_a_s = df_a.sort_values(cols).reset_index(drop=True)
    df_b_s = df_b.sort_values(cols).reset_index(drop=True)

    if df_a_s.equals(df_b_s):
        return True, f"identical ({len(df_a)} rows)"

    # Find which rows differ
    diff_mask = (df_a_s != df_b_s).any(axis=1)
    n_diff    = int(diff_mask.sum())

    # Report up to first 5 differing rows
    sample = []
    for idx in df_a_s.index[diff_mask][:5]:
        row_a = df_a_s.loc[idx].to_dict()
        row_b = df_b_s.loc[idx].to_dict()
        diff_cols = [c for c in cols if row_a[c] != row_b[c]]
        sample.append(f"    row {idx}: differs in columns {diff_cols}")
        for c in diff_cols:
            sample.append(f"      {c}: v1={row_a[c]!r}  v2={row_b[c]!r}")

    return False, (f"{n_diff} rows differ (out of {len(df_a)})\n"
                   + "\n".join(sample))


def compare_txt(path_a, path_b):
    """
    Line-by-line comparison.
    The CELEX ID, version date, and HS vintage lines are kept (they
    should match). Everything else is compared as-is.
    """
    with open(path_a, encoding="utf-8") as f:
        a = f.read().splitlines()
    with open(path_b, encoding="utf-8") as f:
        b = f.read().splitlines()

    if a == b:
        return True, f"identical ({len(a)} lines)"

    # Find first 5 differing lines
    diffs = []
    max_lines = max(len(a), len(b))
    for i in range(max_lines):
        line_a = a[i] if i < len(a) else "<missing>"
        line_b = b[i] if i < len(b) else "<missing>"
        if line_a != line_b:
            diffs.append(f"    line {i+1}:")
            diffs.append(f"      v1   : {line_a!r}")
            diffs.append(f"      v2: {line_b!r}")
            if len(diffs) >= 15:   # 5 differences × 3 lines each
                break

    n_diff_total = sum(1 for i in range(max_lines)
                       if (a[i] if i < len(a) else None) !=
                          (b[i] if i < len(b) else None))
    return False, (f"{n_diff_total} differing lines\n" + "\n".join(diffs))


def compare_one(name, path_a, path_b):
    """Dispatch to CSV or TXT comparison based on extension."""
    ext = name.lower().rsplit(".", 1)[-1]
    if ext == "csv":
        return compare_csv(path_a, path_b)
    elif ext == "txt":
        return compare_txt(path_a, path_b)
    else:
        # Fallback: byte-level comparison
        with open(path_a, "rb") as f: data_a = f.read()
        with open(path_b, "rb") as f: data_b = f.read()
        if data_a == data_b:
            return True, f"identical ({len(data_a)} bytes)"
        return False, f"binary differ (v1={len(data_a)} B, v2={len(data_b)} B)"


# ============================================================
# RUN COMPARISON
# ============================================================

print("=" * 60)
print("OUTPUT FOLDER COMPARISON")
print("=" * 60)
print(f"Version 1   : {FOLDER_V1}")
print(f"Version 2   : {FOLDER_V2}")
print()

files_v1    = list_files(FOLDER_V1)
files_v2    = list_files(FOLDER_V2)

# Map normalised name → real name in each folder
norm_v1    = {normalise_summary_name(n): n for n in files_v1}
norm_v2 = {normalise_summary_name(n): n for n in files_v2}

all_norms = sorted(set(norm_v1) | set(norm_v2))

only_v1    = [norm_v1[n]    for n in all_norms if n in norm_v1    and n not in norm_v2]
only_v2 = [norm_v2[n] for n in all_norms if n in norm_v2 and n not in norm_v1]
in_both      = [n               for n in all_norms if n in norm_v1    and n in norm_v2]

if only_v1:
    print(f"⚠️  Files only in V1 ({len(only_v1)}):")
    for n in only_v1:
        print(f"     {n}")
    print()

if only_v2:
    print(f"⚠️  Files only in V2 ({len(only_v2)}):")
    for n in only_v2:
        print(f"     {n}")
    print()

print(f"Files in both folders: {len(in_both)}")
print()
print("-" * 60)

n_identical    = 0
n_different    = 0
different_list = []

for norm in in_both:
    real_v1    = norm_v1[norm]
    real_v2 = norm_v2[norm]
    path_a = files_v1[real_v1]
    path_b = files_v2[real_v2]

    # Choose the display name (use the real v1 name unless they differ)
    display = real_v1 if real_v1 == real_v2 else f"{real_v1} ↔ {real_v2}"

    try:
        ok, msg = compare_one(norm, path_a, path_b)
    except Exception as e:
        ok, msg = False, f"COMPARISON FAILED: {e}"

    if ok:
        print(f"  ✓  {display:50s} {msg}")
        n_identical += 1
    else:
        print(f"  ✗  {display}")
        # Indent the message by 6 spaces for readability
        for line in msg.splitlines():
            print(f"      {line}")
        n_different += 1
        different_list.append(display)

print()
print("=" * 60)
print("FINAL VERDICT")
print("=" * 60)
print(f"Identical : {n_identical}")
print(f"Different : {n_different}")
if only_v1 or only_v2:
    print(f"Only-v1 : {len(only_v1)}")
    print(f"Only-v2: {len(only_v2)}")
print()

if n_different == 0 and not only_v1 and not only_v2:
    print("✓ ALL FILES IDENTICAL — V1 and V2 produce the same output.")
else:
    print("✗ Differences found — see details above.")
    if different_list:
        print(f"  Differing files: {different_list}")

Additional script: diff harness comparing the per-version outputs of two runs of the extractor

In [ ]:
#!/usr/bin/env python3
# =============================================================================
# RU CN extraction — OLD vs NEW diff harness
# =============================================================================
# Purpose: after re-running the v6 (patched) extractor, compare its per-version
# output against the current (v5) output to enumerate exactly what the fix
# changed — which (annex, version) cells GAIN or LOSE CN codes.
#
# Expected outcome of the empty-lead-cell fix:
#   - Annex XXXVIIIA GAINS 710210, 710231, 710239, 710421, 710491 in every
#     version from 2023-12-19 onward (Parts A/B diamond codes).
#   - Possibly tiny gains in XXVI / XXVII (1-2 codes) — confirm or dismiss.
#   - Everything else: ZERO change. Any other annex that moves is a surprise
#     worth investigating before trusting the new run.
#
# Usage:
#   python RU-CN_extraction_diff_harness.py OLD_DIR NEW_DIR
# where each dir holds per-version files named  RU-CN_extraction_<YYYY-MM-DD>.csv
# (the v5 run folder and the v6 run folder respectively).
# =============================================================================

import sys
import re
import glob
import os
import pandas as pd

FNAME_RX = re.compile(r"RU-CN_extraction_(\d{4}-\d{2}-\d{2})\.csv$")


def load_run(folder):
    """Return {version_date: set((annex, cn_code))} for one extraction run."""
    run = {}
    for path in sorted(glob.glob(os.path.join(folder, "RU-CN_extraction_*.csv"))):
        m = FNAME_RX.search(os.path.basename(path))
        if not m:
            continue
        version = m.group(1)
        df = pd.read_csv(path, dtype=str, encoding="utf-8-sig").fillna("")
        df["annex"]   = df["annex"].str.strip()
        df["cn_code"] = df["cn_code"].str.strip()
        run[version] = set(zip(df["annex"], df["cn_code"]))
    return run


def main(old_dir, new_dir):
    old = load_run(old_dir)
    new = load_run(new_dir)

    old_versions, new_versions = set(old), set(new)
    only_old = sorted(old_versions - new_versions)
    only_new = sorted(new_versions - old_versions)
    common   = sorted(old_versions & new_versions)

    if only_old:
        print(f"WARNING: {len(only_old)} version(s) only in OLD run: {only_old}")
    if only_new:
        print(f"WARNING: {len(only_new)} version(s) only in NEW run: {only_new}")
    print(f"Comparing {len(common)} versions present in both runs.\n")

    # ---- per (annex, version) added / removed ------------------------------
    added_rows, removed_rows = [], []
    for v in common:
        added   = new[v] - old[v]
        removed = old[v] - new[v]
        for annex, code in sorted(added):
            added_rows.append({"version": v, "annex": annex, "cn_code": code})
        for annex, code in sorted(removed):
            removed_rows.append({"version": v, "annex": annex, "cn_code": code})

    df_added   = pd.DataFrame(added_rows)
    df_removed = pd.DataFrame(removed_rows)

    print("=" * 70)
    print("SUMMARY BY ANNEX")
    print("=" * 70)
    if df_added.empty and df_removed.empty:
        print("No differences between the two runs.")
    else:
        annexes = sorted(set(df_added.get("annex", pd.Series(dtype=str)))
                         | set(df_removed.get("annex", pd.Series(dtype=str))))
        for a in annexes:
            a_codes = (sorted(df_added[df_added.annex == a].cn_code.unique())
                       if not df_added.empty else [])
            r_codes = (sorted(df_removed[df_removed.annex == a].cn_code.unique())
                       if not df_removed.empty else [])
            n_ver_added = (df_added[df_added.annex == a].version.nunique()
                           if not df_added.empty else 0)
            print(f"\n{a}")
            if a_codes:
                print(f"  GAINED {len(a_codes)} distinct code(s) across "
                      f"{n_ver_added} version(s): {a_codes}")
            if r_codes:
                print(f"  LOST   {len(r_codes)} distinct code(s): {r_codes}  "
                      f"<-- unexpected, investigate")

    # ---- explicit check of the expected XXXVIIIA recovery ------------------
    print("\n" + "=" * 70)
    print("EXPECTED-RECOVERY CHECK — Annex XXXVIIIA Parts A/B")
    print("=" * 70)
    expected = {"710210", "710231", "710239", "710421", "710491",
                "7102 10", "7102 31", "7102 39", "7104 21", "7104 91"}
    if df_added.empty:
        print("No additions at all — fix did not take effect. Check the run.")
    else:
        got = set(df_added[df_added.annex.str.contains("XXXVIIIA", na=False)].cn_code)
        norm = {c.replace(" ", "") for c in got}
        hit  = sorted(c for c in {"710210", "710231", "710239", "710421", "710491"}
                      if c in norm)
        miss = sorted(c for c in {"710210", "710231", "710239", "710421", "710491"}
                      if c not in norm)
        print(f"  Recovered Part A/B codes: {hit}")
        if miss:
            print(f"  STILL MISSING: {miss}  <-- investigate")
        else:
            print("  All five Part A/B diamond codes recovered. \u2713")

    # ---- write full diffs to disk for the record ---------------------------
    out = os.path.dirname(os.path.abspath(new_dir.rstrip("/"))) or "."
    if not df_added.empty:
        df_added.to_csv(os.path.join(out, "DIFF_added_codes.csv"), index=False)
    if not df_removed.empty:
        df_removed.to_csv(os.path.join(out, "DIFF_removed_codes.csv"), index=False)
    print(f"\nFull diffs written to: {out}/DIFF_added_codes.csv "
          f"(and DIFF_removed_codes.csv if any removals).")

main("/content/RU-CN_extraction_results/RU-CN_results_2026-06-10_16-05-12",  #change the name with the real one!
     "/content/RU-CN_extraction_results/RU-CN_results_2026-06-10_16-12-20")  #change the name with the real one!

In [ ]:
import pandas as pd
d = pd.read_csv("/content/RU-CN_extraction_results/DIFF_added_codes.csv", dtype=str)
x = d[d.annex.str.contains("XVIII", na=False)]
print(x.sort_values(["cn_code","version"]).to_string(index=False))
print()
print(x.groupby("cn_code")["version"].apply(lambda v: sorted(v)).to_string())